In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/

Mounted at /content/drive
/content/drive/MyDrive


In [ ]:
!pip install ta

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29482 sha256=67a6f6abc13167bacce37c55f74611f52ce49ec53815163fe09447f5ccf51a6f
  Stored in directory: /root/.cache/pip/wheels/a1/d7/29/7781cc5eb9a3659d032d7d15bdd0f49d07d2b24fec29f44bc4
Successfully built ta


In [ ]:
import pandas as pd
import numpy as np
from ta.trend import MACD, EMAIndicator
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import BollingerBands, AverageTrueRange

# Load and preprocess data
df = pd.read_csv('./Colab Notebooks/nifty/NIFTY.csv')

# Convert date and ensure numeric columns
df['Day'] = pd.to_datetime(df['Day'], format='%d/%m/%Y')
numeric_cols = ['Advances', 'Declines', 'Open', 'High', 'Low', 'Close']
for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# Sort by date
df.sort_values('Day', inplace=True)

# ======================
# 1. TARGET VARIABLES
# ======================
# Binary Gap Target (original)
df['Next_Open'] = df['Open'].shift(-1)  # Tomorrow's open price
df['Today_Close'] = df['Close']  # Today's close (for gap calculation)
gap_threshold = 0.0001 * df['Today_Close']
df['Gap'] = np.where(
    df['Next_Open'] > (df['Today_Close'] + gap_threshold),
    1,  # Tomorrow will Gap Up
    0   # Tomorrow will Gap Down (including no significant gap)
)

# New Multi-Class 5-Day Movement Target
df['Future_Close'] = df['Close'].shift(-5)  # Close price 5 days later
movement_threshold = 0.005  # 0.5% threshold for significant movement

df['5d_Movement'] = np.select(
    [
        (df['Future_Close'] > df['Close'] * (1 + movement_threshold)),  # Positive movement
        (df['Future_Close'] < df['Close'] * (1 - movement_threshold)),  # Negative movement
    ],
    [
        1,   # Positive movement
        -1,  # Negative movement
    ],
    default=0  # No significant movement
)

# ======================
# 2. ADD OSCILLATORS (Acosc & Awosc)
# ======================
def calculate_awesome_oscillator(df, short_period=5, long_period=34):
    midpoint = (df['High'] + df['Low']) / 2
    short_sma = midpoint.rolling(window=short_period).mean()
    long_sma = midpoint.rolling(window=long_period).mean()
    return short_sma - long_sma

def calculate_accelerator_oscillator(df, short_period=5, long_period=34, signal_period=5):
    awosc = calculate_awesome_oscillator(df, short_period, long_period)
    awosc_sma = awosc.rolling(window=signal_period).mean()
    return awosc - awosc_sma

df['Awosc'] = calculate_awesome_oscillator(df)
df['Acosc'] = calculate_accelerator_oscillator(df)

# ======================
# 3. ADD TECHNICAL FEATURES
# ======================
# Trend Indicators
df['SMA_5'] = df['Close'].rolling(5).mean()
df['SMA_20'] = df['Close'].rolling(20).mean()
df['EMA_12'] = EMAIndicator(df['Close'], window=12).ema_indicator()
df['EMA_26'] = EMAIndicator(df['Close'], window=26).ema_indicator()
df['MACD'] = MACD(df['Close']).macd()

# Momentum Indicators
df['RSI_14'] = RSIIndicator(df['Close'], window=14).rsi()
df['Stoch_%K'] = StochasticOscillator(df['High'], df['Low'], df['Close'], window=14).stoch()

# Volatility Indicators
bb = BollingerBands(df['Close'], window=20)
df['BB_Upper'] = bb.bollinger_hband()
df['BB_Lower'] = bb.bollinger_lband()
df['ATR_14'] = AverageTrueRange(df['High'], df['Low'], df['Close'], window=14).average_true_range()

# Price Action Features
df['Body_Size'] = (df['Close'] - df['Open']).abs() / df['Open']  # Candle body size
df['High_Low_Range'] = (df['High'] - df['Low']) / df['Open']  # Daily range
df['Close_Position'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'])  # Close position in daily range

# Market Breadth
df['AD_Ratio'] = df['Advances'] / df['Declines']
df['Advance_Decline_Diff'] = df['Advances'] - df['Declines']

# Lagged Features
for i in [1, 2, 3, 5]:  # Added 5-day lags for the new target
    df[f'Return_{i}d'] = df['Close'].pct_change(i)
    df[f'Volume_{i}d'] = df['Advances'].shift(i)

# ======================
# 4. CLEANUP AND SAVE
# ======================
# Calculate daily returns and volatility
df['Daily_Return'] = df['Close'].pct_change()
df['Rolling_Vol_5'] = df['Daily_Return'].rolling(5).std()

# Drop rows with missing values (from rolling calculations and shifts)
df.dropna(inplace=True)

# Remove temporary columns
df.drop(columns=['Next_Open', 'Future_Close'], inplace=True)

# Save to new CSV
df.to_csv('./Colab Notebooks/nifty/NIFTY_with_MultiClass_Targets.csv', index=False)

# Show target distributions
print("\nGap Distribution (Binary):")
print(df['Gap'].value_counts())
print("\n5-Day Movement Distribution (Multi-class):")
print(df['5d_Movement'].value_counts())

# Show sample of the final data
print("\nSample Data:")
print(df[['Day', 'Close', 'Gap', '5d_Movement', 'Awosc', 'Acosc']].tail(10))


Gap Distribution (Binary):
Gap
1    1001
0     487
Name: count, dtype: int64

5-Day Movement Distribution (Multi-class):
5d_Movement
 1    723
-1    495
 0    270
Name: count, dtype: int64

Sample Data:
            Day     Close  Gap  5d_Movement       Awosc       Acosc
1515 2025-02-07  23559.95    0           -1  110.552206  120.018735
1516 2025-02-10  23381.60    0           -1  148.504265   88.424706
1517 2025-02-11  23071.80    0           -1   84.222500   -8.030500
1518 2025-02-12  23045.25    1            0  -43.829853 -120.976265
1519 2025-02-13  23031.40    1           -1 -135.114118 -167.981118
1520 2025-02-14  22929.25    0           -1 -239.768088 -202.571029
1521 2025-02-17  22959.50    1           -1 -333.499706 -199.901853
1522 2025-02-18  22945.30    0           -1 -359.648382 -137.276353
1523 2025-02-19  22932.90    0           -1 -333.537353  -53.223824
1524 2025-02-20  22913.15    0           -1 -354.628971  -30.412471
